<a href="https://colab.research.google.com/github/mnsbharadwaj/AI-NLP/blob/master/RNN_based_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import requests

# 1. Download and load the Tiny Shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text

# Explanation:
# - `import torch`: Imports PyTorch for building and training the neural network.
# - `import torch.nn as nn`: Imports neural network modules from PyTorch.
# - `import numpy as np`: Imports NumPy for numerical operations.
# - `import requests`: Imports requests library to download the dataset.
# - `url`: URL to the Tiny Shakespeare dataset (a text file hosted by Andrej Karpathy).
# - `response = requests.get(url)`: Fetches the dataset from the URL.
# - `text = response.text`: Stores the dataset as a string.

# 2. Preprocess the dataset
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# Convert text to indices
data = [char_to_idx[ch] for ch in text]

# Explanation:
# - `chars = sorted(list(set(text)))`: Creates a sorted list of unique characters in the dataset.
# - `vocab_size = len(chars)`: Number of unique characters (vocabulary size).
# - `char_to_idx`: Dictionary mapping each character to a unique index (e.g., 'a' -> 0).
# - `idx_to_char`: Reverse mapping from index to character (e.g., 0 -> 'a').
# - `data`: Converts the entire text into a list of indices corresponding to characters.

# 3. Create input-output pairs for training
seq_length = 25
X, y = [], []
for i in range(0, len(data) - seq_length):
    X.append(data[i:i + seq_length])
    y.append(data[i + 1:i + seq_length + 1])
X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

# Explanation:
# - `seq_length = 25`: Defines the length of input sequences (25 characters).
# - `X, y = [], []`: Initializes lists to store input and target sequences.
# - Loop creates pairs: X[i] is a sequence of `seq_length` characters, y[i] is the next `seq_length` characters (shifted by 1).
# - `torch.tensor`: Converts lists to PyTorch tensors with dtype `long` for indices.

# 4. Define the RNN model
class TinyLLM(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden

# Explanation:
# - `class TinyLLM(nn.Module)`: Defines a custom RNN model inheriting from `nn.Module`.
# - `__init__`: Initializes the model layers.
#   - `nn.Embedding(vocab_size, hidden_size)`: Converts character indices to dense vectors.
#   - `nn.RNN(hidden_size, hidden_size, num_layers, batch_first=True)`: RNN layer with `hidden_size` units, `num_layers` stacked RNNs, and `batch_first=True` for input shape (batch, seq_len, features).
#   - `nn.Linear(hidden_size, vocab_size)`: Linear layer to map RNN outputs to vocabulary size (for predicting next character).
# - `forward`: Defines the forward pass.
#   - Embeds input indices, passes through RNN, applies linear layer, returns output and hidden state.

# 5. Model parameters and setup
hidden_size = 128
num_layers = 1
model = TinyLLM(vocab_size, hidden_size, num_layers)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Explanation:
# - `hidden_size = 128`: Size of the hidden state in the RNN.
# - `num_layers = 1`: Single RNN layer for simplicity.
# - `model = TinyLLM(...)`: Instantiates the model.
# - `criterion = nn.CrossEntropyLoss()`: Loss function for multi-class classification (predicting next character).
# - `optimizer = torch.optim.Adam(...)`: Adam optimizer with learning rate 0.001.

# 6. Training loop
num_epochs = 2  # Limited for demonstration
batch_size = 64
model.train()
for epoch in range(num_epochs):
    for i in range(0, X.size(0), batch_size):
        inputs = X[i:i + batch_size]
        targets = y[i:i + batch_size]

        optimizer.zero_grad()
        output, _ = model(inputs)
        loss = criterion(output.view(-1, vocab_size), targets.view(-1))
        loss.backward()
        optimizer.step()

        if i % 1000 == 0:
            print(f'Epoch {epoch+1}, Step {i//batch_size}, Loss: {loss.item():.4f}')

# Explanation:
# - `num_epochs = 2`: Trains for 2 epochs (limited for brevity; increase for better results).
# - `batch_size = 64`: Processes 64 sequences at a time.
# - `model.train()`: Sets model to training mode.
# - Loop over batches:
#   - `inputs = X[i:i + batch_size]`: Gets a batch of input sequences.
#   - `targets = y[i:i + batch_size]`: Gets corresponding target sequences.
#   - `optimizer.zero_grad()`: Clears previous gradients.
#   - `output, _ = model(inputs)`: Forward pass to get predictions.
#   - `loss = criterion(...)`: Computes loss by reshaping output and targets for CrossEntropyLoss.
#   - `loss.backward()`: Computes gradients.
#   - `optimizer.step()`: Updates model parameters.
#   - Prints loss every 1000 steps.

# 7. Generate text
def generate_text(model, seed_text, length=100):
    model.eval()
    hidden = None
    input_seq = torch.tensor([char_to_idx[ch] for ch in seed_text], dtype=torch.long).unsqueeze(0)

    generated = seed_text
    for _ in range(length):
        output, hidden = model(input_seq, hidden)
        probs = torch.softmax(output[:, -1, :], dim=-1)
        next_char_idx = torch.multinomial(probs, num_samples=1).item()
        generated += idx_to_char[next_char_idx]
        input_seq = torch.tensor([[next_char_idx]], dtype=torch.long)

    return generated

# Explanation:
# - `generate_text`: Function to generate text given a seed and desired length.
# - `model.eval()`: Sets model to evaluation mode (disables dropout, etc.).
# - `hidden = None`: Initializes hidden state as None (RNN will initialize internally).
# - `input_seq`: Converts seed text to tensor of indices, with batch dimension.
# - Loop generates `length` characters:
#   - Passes input through model to get output and updated hidden state.
#   - `torch.softmax`: Converts last output to probabilities.
#   - `torch.multinomial`: Samples next character index based on probabilities.
#   - Appends character to `generated` string and updates `input_seq`.
# - Returns generated text.

# 8. Generate sample output
seed = "To be or not to be"
generated_text = generate_text(model, seed, length=100)
print("\nGenerated Text:")
print(generated_text)

# Explanation:
# - `seed`: Starting text for generation.
# - Calls `generate_text` to produce 100 characters starting from the seed.
# - Prints the generated text.

Epoch 1, Step 0, Loss: 4.2304
Epoch 1, Step 125, Loss: 2.5095
Epoch 1, Step 250, Loss: 2.5892
Epoch 1, Step 375, Loss: 2.0633
Epoch 1, Step 500, Loss: 2.0590
Epoch 1, Step 625, Loss: 1.6741
Epoch 1, Step 750, Loss: 2.0029
Epoch 1, Step 875, Loss: 2.2998
Epoch 1, Step 1000, Loss: 2.0650
Epoch 1, Step 1125, Loss: 2.1100
Epoch 1, Step 1250, Loss: 1.7497
Epoch 1, Step 1375, Loss: 1.8911
Epoch 1, Step 1500, Loss: 1.8696
Epoch 1, Step 1625, Loss: 2.0733
Epoch 1, Step 1750, Loss: 1.8955
Epoch 1, Step 1875, Loss: 1.9909
Epoch 1, Step 2000, Loss: 1.8231
Epoch 1, Step 2125, Loss: 2.1779
Epoch 1, Step 2250, Loss: 1.7954
Epoch 1, Step 2375, Loss: 2.1714
Epoch 1, Step 2500, Loss: 2.2530
Epoch 1, Step 2625, Loss: 1.6668
Epoch 1, Step 2750, Loss: 1.8338
Epoch 1, Step 2875, Loss: 2.1178
Epoch 1, Step 3000, Loss: 1.6497
Epoch 1, Step 3125, Loss: 1.7614
Epoch 1, Step 3250, Loss: 2.2604
Epoch 1, Step 3375, Loss: 1.4628
Epoch 1, Step 3500, Loss: 1.5863
Epoch 1, Step 3625, Loss: 1.7184
Epoch 1, Step 3750, 